# Football Match Result Prediction with MLP — 2026 FIFA World Cup

**Course:** PSI0325 — Fundamentals of Adaptation and Machine Learning
**Student:** Jefferson Santos Monteiro (NUSP 15527878)

This notebook is the final deliverable for the course project. It reproduces
the full pipeline: dataset construction, model training (MLP and a linear
baseline), evaluation against the real 2026 FIFA World Cup matches, and
predictions for hypothetical matchups.

Run this notebook from the `notebooks/` folder with the repository's
virtual environment active. Make sure `data/raw/results.csv` exists first
(see the repository README for the download link) — everything else is
regenerated by this notebook.

## 1. Problem statement

The goal is to train a multilayer perceptron (MLP) that estimates
P(home win), P(draw), and P(away win) for international football matches,
using a 3-neuron softmax output trained with cross-entropy loss.

**Validation strategy (the core contribution of this project):** instead of
a random train/test split, the model is trained only on matches **before**
the 2026 FIFA World Cup and tested directly on the **104 real matches** of
that tournament, whose outcomes are already known. This tests not only
whether the model predicts the right outcome, but whether its predicted
probabilities are well **calibrated** against a full, already-completed
tournament — which a random split could not evaluate.

## 2. Setup

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 3. Dataset and feature engineering

Source: the public Kaggle dataset *"International football results from
1872 to 2026"* (`results.csv`), which includes a community-maintained
extension covering the 2026 World Cup matches.

Three engineered features are used, each computed strictly from information
available **before** each match (no data leakage):

- **`elo_diff`** — Elo rating difference (home − away). Elo is computed
  *sequentially over the entire history since 1872* (not reset in 2000),
  so ratings are not artificially flattened at the start of the training
  window.
- **`home_flag`** — home advantage indicator (0 for neutral-venue matches).
- **`form_diff`** — difference in points earned over each team's last 10
  matches (home − away).
- **`elo_home_interaction`** — `elo_diff × home_flag`, added to test
  whether home advantage depends on the strength gap between teams (a
  non-linear interaction that a purely linear model cannot represent).

**Split:** training = matches from 2000-01-01 to the eve of the 2026 World
Cup (2026-06-10); test = the 104 matches of the tournament itself
(2026-06-11 to 2026-07-19). The split is temporal, not random, so the model
never has access to future information during training.

The cell below regenerates `data/processed/train.csv`, `test.csv`, and
`team_state.csv` (each team's most recent Elo and form, used later for
predictions) from the raw dataset.

In [ ]:
import build_dataset

build_dataset.main()

In [ ]:
train_df = pd.read_csv("../data/processed/train.csv")
test_df = pd.read_csv("../data/processed/test.csv")

print(f"Training matches: {len(train_df)}  ({train_df['date'].min()} to {train_df['date'].max()})")
print(f"Test matches (2026 World Cup): {len(test_df)}")
train_df.head()

In [ ]:
train_df["result"].value_counts(normalize=True).rename(
    index={0: "home_win", 1: "draw", 2: "away_win"}
).rename("proportion")

## 4. Model

### 4.1 MLP architecture

A single hidden layer with a small number of units was chosen deliberately.
With only 4 input features and roughly 25k training examples, a wide/deep
network offers little extra representational power and mainly increases
overfitting risk — this is corroborated by the linear-vs-MLP comparison in
Section 5. Kaiming (He) initialization is used for the ReLU hidden layer,
and Xavier initialization for the output layer that feeds into softmax.
Dropout (0.2) and L2 weight decay are used for regularization.

### 4.2 Linear baseline

To justify the use of non-linearity (as required by the course), a linear
baseline — logistic regression expressed as a single linear layer with no
hidden layer — is trained on the same features and compared against the MLP
using the same cross-entropy metric on the World Cup test set.

In [ ]:
from model import MatchOutcomeMLP
from model_linear import LinearBaseline

print(MatchOutcomeMLP())
print()
print(LinearBaseline())

## 5. Training

Both models are trained with the Adam optimizer and cross-entropy loss.
A chronological 10% slice of the training set (the most recent matches
before the split) is held out purely to monitor overfitting during
training — this is separate from, and never used to tune against, the
World Cup test set.

In [ ]:
import train as train_mlp

train_mlp.main()

In [ ]:
display(Image("../outputs/figures/learning_curve.png"))

In [ ]:
import train_linear

train_linear.main()

## 6. Evaluation on the 2026 World Cup

The primary metric is **cross-entropy** on the 104 real World Cup matches
(measuring probability quality, not just correctness), with accuracy
reported only as a secondary reference. The calibration curve is the key
result for this project's stated validation approach.

In [ ]:
import evaluate

evaluate.main()

In [ ]:
display(Image("../outputs/figures/calibration_curve.png"))

In [ ]:
display(Image("../outputs/figures/confusion_matrix.png"))

In [ ]:
display(Image("../outputs/figures/roc_curves.png"))

### 6.1 Discussion

- **Calibration:** the home-win and away-win curves track the diagonal
  reasonably well, indicating the predicted probabilities are broadly
  trustworthy for those outcomes.
- **Draws are the clear limitation:** the confusion matrix shows the model
  never predicts a draw as the most likely outcome, and the one-vs-rest
  ROC curve for `draw` has AUC ≈ 0.50 — no better than chance. This mirrors
  a well-known difficulty in the football-prediction literature: draws lack
  a clear "signature" in strength-difference features like Elo, since they
  arise from more subtle factors (playing style, game context) that are not
  captured by `elo_diff`, `home_flag`, or `form_diff`. This limitation is
  precisely what the calibration-based evaluation was designed to surface —
  a plain accuracy score would have hidden it.

## 7. Linear baseline vs. MLP — does non-linearity help?

This is run last, after both models are trained, so the comparison reflects
the final architecture (including the `elo_home_interaction` feature).

In [ ]:
import compare_models

compare_models.main()

Interpret this table together with Section 6: if the MLP's cross-entropy is
at or below the linear baseline's, that supports the case for non-linearity
(helped here by the `elo_diff × home_flag` interaction term, which a linear
model cannot represent). If the two are close, that is itself a valid and
informative finding — it suggests that, with this compact 4-feature
representation, the relationship between inputs and outcome is close to
linear, and a much larger network would likely just overfit without
improving generalization to the World Cup.

## 8. Predicting hypothetical matchups

Using each team's most recent Elo rating and recent-form points
(`data/processed/team_state.csv`), the trained MLP can estimate outcome
probabilities for any hypothetical matchup, not just matches already in the
dataset.

In [ ]:
import predict

for home, away, neutral in [
    ("Brazil", "Spain", False),
    ("Brazil", "Germany", False),
    ("Argentina", "Spain", True),
]:
    predict.predict_and_print(home, away, neutral=neutral)
    print()

## 8.5 Selected World Cup matches — model vs. actual results

A curated comparison table for presentation purposes: predicted
probabilities vs. actual outcomes for six notable matches from the
tournament, generated directly from the test-set predictions (not
re-derived from `team_state.csv`, so the features match exactly what
the model saw during evaluation).

In [ ]:
import generate_match_table

generate_match_table.main()

In [ ]:
pd.read_csv("../outputs/predictions_selected.csv")

## 9. Conclusion

This project trained an MLP to estimate match outcome probabilities for the
2026 FIFA World Cup, validated against the tournament's real results rather
than a random split. The model produces reasonably calibrated probabilities
for decisive outcomes (home/away win) but fails to discriminate draws — a
limitation consistent with prior football-prediction literature and
directly exposed by the calibration-based evaluation this project was
designed around. A linear baseline and an explicit interaction feature were
used to test, rather than assume, whether the MLP's non-linearity was
warranted for this compact feature set.